In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances
from geopy.geocoders import Nominatim

/opt/anaconda3/envs/LIGO/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
full_df = pd.read_csv("umn_apartment_data.csv") #reading csv file

#spliting dataframes between amenities and other data points
df = full_df.iloc[:,:]
amenities_df = full_df.iloc[:,9:]

#loop through all amenities to put into 1 list
temp_df = pd.DataFrame()
for index, values in amenities_df.iterrows():
    my_list = [values.unique()[:-1]]
    temp_df = pd.concat([temp_df, pd.Series(my_list)], ignore_index = True) #adding list into dataframe in each iteration

#recombining main df with modified amenities df
df = pd.concat([df, temp_df], axis = 1, ignore_index = True)

#renaming columns
column_list = ["Name", "Floor Plan", "Address", "Bedrooms", "Bathrooms", "Price", "Size", "Availability", "Link", "Amenities", "AmenitiesID"]
column_dic = {}
for i in range(len(column_list)):
    column_dic[df.columns.values[i]] = column_list[i]

df = df.rename(columns = column_dic)

In [39]:
df[df["Name"] == "808 Berry Place"]

,Name,Floor Plan,Address,Bedrooms,Bathrooms,Price,Size,Availability,Link,Amenities,AmenitiesID
327,808 Berry Place,Three Bedroom A,808 Berry St St Paul MN 55114 USA,4,2.0,3072.0,1470.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
328,808 Berry Place,Three Bedroom B,808 Berry St St Paul MN 55114 USA,3,2.0,2865.0,1532.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
329,808 Berry Place,Three Bedroom Townhome A,808 Berry St St Paul MN 55114 USA,3,2.5,3061.0,1794.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
330,808 Berry Place,Two Bedroom A,808 Berry St St Paul MN 55114 USA,2,2.0,2160.0,1076.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
331,808 Berry Place,Two Bedroom C,808 Berry St St Paul MN 55114 USA,2,2.0,2270.0,1121.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
332,808 Berry Place,Two Bedroom D,808 Berry St St Paul MN 55114 USA,2,2.0,2200.0,1171.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
333,808 Berry Place,Two Bedroom E,808 Berry St St Paul MN 55114 USA,2,2.0,2290.0,1181.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
334,808 Berry Place,Two Bedroom Loft A,808 Berry St St Paul MN 55114 USA,2,2.0,2382.0,1361.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
335,808 Berry Place,Two Bedroom Townhome B,808 Berry St St Paul MN 55114 USA,2,2.5,2407.0,1430.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]
336,808 Berry Place,One Bedroom A,808 Berry St St Paul MN 55114 USA,1,1.0,1708.0,740.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"High Ceilings, Stainless Steel Kitchen Applian...",[]


In [38]:
df.iloc[327,3] = 4

Potential Features:
- Location (Distance from a Central Point)
- Amount of Bathrooms/Beds
- Price
- Size
- Amenities (Specific ID for Each Kind)
- Shaping (potentially)

In [3]:
#replacing each value in Bedrooms column with numerical value
idx = 0
for value in df["Bedrooms"]:
    if value == "Studio":
        df.iloc[idx, 3] = "0"
    else:
        df.iloc[idx, 3] = value[0]
    idx += 1

#replacing each value in Price column with numerical value
idx = 0
for value in df["Price"]:
    end_id = value.find("-")
    try:
        df.iloc[idx, 5] = pd.to_numeric(value[1:end_id])
    except:
        df.iloc[idx, 5] = None
    idx += 1

#if price is not divided for each indivudal, do it manually
# ids = df[df["Price"] > 2000].index
# df.iloc[ids, 5] /= 4


#replacing each value in Size column with numerical value
# idx = 0
# for value in df["Size"]:
#     if type(value) is str:
#         end_id = value.find("-") + 1
#     try:
#         df.iloc[idx, 6] = pd.to_numeric(value[1:end_id])
#     except:
#         df.iloc[idx, 6] = None
#     idx += 1


#making each value numeric - if error occurs replaces value with NaN
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Size'] = pd.to_numeric(df['Size'], errors='coerce')
df['Bathrooms'] = pd.to_numeric(df['Bathrooms'], errors='coerce')
df['Bedrooms'] = pd.to_numeric(df['Bedrooms'], errors='coerce')

df = df.dropna().reset_index(drop = True) #removing all none types

In [4]:
print(df[["Price", "Size", "Bathrooms", "Bedrooms"]].corr()) #correlation square

              Price      Size  Bathrooms  Bedrooms
Price      1.000000  0.642045   0.411153  0.531827
Size       0.642045  1.000000   0.623007  0.676832
Bathrooms  0.411153  0.623007   1.000000  0.723046
Bedrooms   0.531827  0.676832   0.723046  1.000000


In [ ]:
full_house_df = pd.read_csv("umn_housing_data.csv") #reading csv

In [4]:
#spliting dataframes between amenities and other data points
house_df = full_house_df.iloc[:,:7]
house_amenities_df = full_house_df.iloc[:,7:]
house_amenities_df

#loop through all amenities to put into 1 list
temp_df = pd.DataFrame()
for index, values in house_amenities_df.iterrows():
    my_list = [values.unique()[:-1]]
    temp_df = pd.concat([temp_df, pd.Series(my_list)], ignore_index = True)

#recombining main df with modified amenities df
house_df = pd.concat([house_df, temp_df], axis = 1, ignore_index = True)

#naming columns
column_list = ["Address", "Name", "Bedrooms", "Bathrooms", "Price", "Size", "Availability", "Amenities"]
column_dic = {}
for i in range(len(column_list)):
    column_dic[house_df.columns.values[i]] = column_list[i]

house_df = house_df.rename(columns = column_dic)
house_df #outputs dataframe



NameError: name 'full_house_df' is not defined

In [ ]:
#creates all values in Price column numeric
idx = 0
for value in house_df["Price"]:
    end_id = value.find("-")
    try:
        house_df.iloc[idx, 4] = pd.to_numeric(value[1:end_id])
    except:
        house_df.iloc[idx, 4] = None
    idx += 1

In [4]:
df[['Bedrooms', 'Bathrooms', 'Size', 'Price']]

,Bedrooms,Bathrooms,Size,Price
0,4,4.0,1239.0,599.0
1,4,4.0,1239.0,639.0
2,3,3.0,1035.0,669.0
3,3,3.0,1035.0,719.0
4,2,2.0,806.0,849.0
...,...,...,...,...
414,2,1.0,743.0,1425.0
415,5,2.0,2600.0,3295.0
416,5,4.0,1600.0,3295.0
417,5,4.0,0.0,3095.0


Alternatives to Cosine Similarity:
- Euclidean Distance
- Dot Product

In [40]:
id = 0 #choosing the id of the apartment you want to find recommendations for
req = df.iloc[id,3] #grabbing # of bedrooms in given id

#defining df of all numeric values that has the same amount of bedrooms as a given id
cdf = df[df['Bedrooms'] == req].copy() 
cdf = cdf[['Bedrooms', 'Bathrooms', 'Size', 'Price']]

#adding weights to each option
cdf["Bathrooms"] *= 1.5
cdf["Price"] /= 500
cdf["Size"] /= 1000

#calculating cosine similarity
user_similarity = cosine_similarity(cdf)

#finding similar apartments to given id
apartment_id = cdf.index.get_loc(id)
similar_apartments = user_similarity[id]

similar_apartments_indices = np.argsort(similar_apartments)[::-1][1:6] #shows best 5 similar apartments (ignoring the apartment itself)
similar_apartments = cdf.index[similar_apartments_indices] #grabbing indexes of similar apartments

df.loc[similar_apartments]

,Name,Floor Plan,Address,Bedrooms,Bathrooms,Price,Size,Availability,Link,Amenities,AmenitiesID
1,The Quad on Delaware,D1R - Renovated 4 Bedroom 4 Bath,2508 Delaware Street SE Minneapolis MN 55414,4,4.0,639.0,1239.0,08-27-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"Garbage Disposal, Parking Available, 24 Hour E...",[]
38,The Knoll Dinkytown,D1,1101 University Ave SE Minneapolis MN 55414,4,4.0,955.0,1596.0,08-19-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"Custom Tile Backsplash, Furnished, Granite Cou...",[]
51,The Bridges Dinkytown,D1,930 University Ave SE Minneapolis MN 55414,4,4.0,955.0,1626.0,08-21-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"Custom Tile Backsplash, Furnished, Granite Cou...",[]
39,The Knoll Dinkytown,D3,1101 University Ave SE Minneapolis MN 55414,4,4.0,975.0,1657.0,08-19-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"Custom Tile Backsplash, Furnished, Granite Cou...",[]
168,Factory Lofts,4 Bedroom - Individual Bedroom,2655 Essex St SE Minneapolis MN 55414 USA,4,4.0,694.0,300.0,Now,https://listings.umn.edu/city/minneapolis-mn/l...,"14 Foot Ceilings, Air Conditioning, Ample Clos...",[]


In [5]:
id = 0 #choosing the id of the apartment you want to find recommendations for
req = df.iloc[id,3] #grabbing # of bedrooms in given id

#defining df of all numeric values that has the same amount of bedrooms as a given id
edf = df[df['Bedrooms'] == req].copy().drop_duplicates("Address")
edf = edf[['Address', 'Bedrooms', 'Bathrooms', 'Size', 'Price']]

geolocator = Nominatim(user_agent="apartment-finder") 

central_address = "207 Church Street SE, Minneapolis, MN 55455" #Lind Hall
central_location = geolocator.geocode(central_address) #grabs location

address_list = []

lat_list = []
long_list = []

for address in edf["Address"]:
    time.sleep(1) #Nominatim allow max 1 inquiry per second, prevents timeout error
    #can use a rotating api key to bypass rate limit - but need something that uses api keys
    
    #if address is not already used grab the coordinates of the address
    if address not in address_list: 
        location = geolocator.geocode(address)
        address_list.append(address) #to check if it has been used
        
    try:
        lat_list.append(location.latitude)
        long_list.append(location.longitude)
    except (AttributeError): #will edit later
        lat_list.append(-1)
        long_list.append(-1)

distance_list = []

for i in range(len(lat_list)):
    distance = ((central_location.latitude - lat_list[i])**2 + (central_location.latitude - lat_list[i])**2)**0.5 #calculating distance between Lind Hall and given apartment
    distance_list.append(distance * 10**5 * 1.11 / 1609.34) #distance in miles

cdf = pd.concat([edf.reset_index(), pd.Series(lat_list), pd.Series(long_list), pd.Series(distance_list)], axis=1)

cdf = cdf[["index", "Bedrooms", "Bathrooms", "Size", "Price", 2]]
cdf.columns.values[-1] = "Distance"

# #adding weights to each option
# cdf["Bathrooms"] *= 1.5
# cdf["Price"] /= 500
# cdf["Size"] /= 1000
# cdf["Distance"] *= 2


In [12]:
df[df["Name"] == "808 Berry Place"].index[0]

np.int64(327)

In [84]:
test = cdf.copy()

In [85]:
cdf = test

cdf["Bathrooms"] *= 5
cdf["Price"] /= 20
cdf["Size"] /= 500
cdf["Distance"] *= 40

#calculating cosine similarity
user_similarity = euclidean_distances(cdf)

#finding similar apartments to given id
apartment_id = cdf.index.get_loc(id)
similar_apartments = user_similarity[id]

similar_apartments_indices = np.argsort(similar_apartments)[1:] #shows best 5 similar apartments (ignoring the apartment itself)
similar_apartments = cdf.index[similar_apartments_indices] #grabbing indexes of similar apartments

df.loc[cdf.loc[similar_apartments]["index"]]

,Name,Floor Plan,Address,Bedrooms,Bathrooms,Price,Size,Availability,Link,Amenities,AmenitiesID
19,Bierman Place,The Quad - By-the-bed,1401 6th Street SE Minneapolis MN 55414 USA,4,2.0,650.0,0.0,Call for Availability,https://listings.umn.edu/city/minneapolis-mn/l...,"Bedrooms Keyed Separately, Breakfast Bar, Dish...",[]
38,The Knoll Dinkytown,D1,1101 University Ave SE Minneapolis MN 55414,4,4.0,955.0,1596.0,08-19-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"Custom Tile Backsplash, Furnished, Granite Cou...",[]
51,The Bridges Dinkytown,D1,930 University Ave SE Minneapolis MN 55414,4,4.0,955.0,1626.0,08-21-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"Custom Tile Backsplash, Furnished, Granite Cou...",[]
74,University Village,C - Private Bedroom,2515 University Ave SE Minneapolis MN 55414,4,2.0,750.0,1261.0,Now,https://listings.umn.edu/city/minneapolis-mn/l...,"Air Conditioning, Ample Closet Space, Carpet, ...",[]
26,525 Tenth Apartments Dinkytown,Unit 205,525 10th Ave SE Minneapolis MN 55414 USA,4,2.0,2796.0,1282.0,08-31-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"9 Foot Ceilings, Air Conditioning, Ample Close...",[]
28,Spectrum Apartments & Townhomes,809-8,815 SE 9th Ave Minneapolis MN 55414 USA,4,4.0,2895.0,1715.0,06-05-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"9 Foot Ceilings, Air Conditioning, Bedrooms Ke...",[]
6,Doyle Apartments,707,1307 4th Street SE Minneapolis MN 55414 USA,4,2.0,3180.0,1192.0,08-25-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"9 Foot Ceilings, Air Conditioning, Bedrooms Ke...",[]
102,Chateau Student Housing Cooperative,907,425 13th Ave SE Minneapolis MN 55414 USA,4,2.0,2445.0,1100.0,Now,https://listings.umn.edu/city/minneapolis-mn/l...,"Air Conditioning, Ample Closet Space, Bedrooms...",[]
93,Golden Lodge,The Swede,621 15th Ave SE Minneapolis MN 55414,4,3.0,2780.0,1600.0,09-01-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"Air Conditioning, Bedrooms Keyed Separately, B...",[]
163,501 6th St SE,#201,501 6th St SE Minneapolis MN 55414 USA,4,2.0,2550.0,1600.0,09-01-2026,https://listings.umn.edu/city/minneapolis-mn/l...,"Air Conditioning, Den, Dishwasher, Fresh Paint...",[]


In [86]:
test

,index,Bedrooms,Bathrooms,Size,Price,Distance
0,0,4,20.0,2.478,29.95,11.341755
1,6,4,10.0,2.384,159.00,25.696377
2,19,4,10.0,0.000,32.50,31.480597
3,26,4,10.0,2.564,139.80,37.038522
4,28,4,20.0,3.430,144.75,52.223418
5,38,4,20.0,3.192,47.75,13.159931
6,51,4,20.0,3.252,47.75,20.380746
7,74,4,10.0,2.522,37.50,4.406932
8,93,4,15.0,3.200,139.00,29.965970
9,102,4,10.0,2.200,122.25,28.807566


In [89]:
location = geolocator.geocode("2828 University Ave SE, Minneapolis, MN 55414")

In [90]:
location.latitude, location.longitude

(44.9710283, -93.2161298)